In [1]:
!pip install pandas numpy scikit-learn tensorflow

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [ ]:
df = pd.read_csv('batterydegradation.csv')


In [ ]:
df['vol_rolling_mean'] = df.groupby('battery_id')['voltage'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['temp_rolling_mean'] = df.groupby('battery_id')['temperature'].transform(lambda x: x.rolling(3, min_periods=1).mean())

features = ['cycle', 'voltage', 'temperature', 'vol_rolling_mean', 'temp_rolling_mean']
target = 'capacity'

X = df[features]
y = df[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
pred_rf = rf.predict(X_test_scaled)

In [ ]:
gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42)
gbr.fit(X_train_scaled, y_train)
pred_gbr = gbr.predict(X_test_scaled)

In [ ]:
X_train_lstm = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test_lstm = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

lstm = Sequential([
    tf.keras.Input(shape=(1, X_train_scaled.shape[1])),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])
lstm.compile(optimizer='adam', loss='mse')
lstm.fit(X_train_lstm, y_train, epochs=50, batch_size=16, verbose=0)
pred_lstm = lstm.predict(X_test_lstm).flatten()

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step


In [ ]:
def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"=== {name} ===")
    print(f"RMSE : {rmse:.4f} Ah")
    print(f"MAE  : {mae:.4f} Ah")
    print(f"R2   : {r2:.4f}\n")

evaluate("Random Forest Regressor", y_test, pred_rf)
evaluate("Gradient Boosting Regressor", y_test, pred_gbr)
evaluate("LSTM Deep Learning", y_test, pred_lstm)

=== Random Forest Regressor ===
RMSE : 0.1143 Ah
MAE  : 0.0408 Ah
R2   : 0.8752

=== Gradient Boosting Regressor ===
RMSE : 0.0955 Ah
MAE  : 0.0475 Ah
R2   : 0.9129

=== LSTM Deep Learning ===
RMSE : 0.2078 Ah
MAE  : 0.1140 Ah
R2   : 0.5875

